In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 84.1 MB/s eta 0:00:00


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import faiss
import numpy as np

embeddings = np.load("/content/drive/MyDrive/copydays-ndid/NDID_6000_image_data_set/embeddings/clip_embeddings.npy").astype("float32")

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)   # cosine similarity (after normalization)
index.add(embeddings)

faiss.write_index(index, "ndid_faiss.index")

In [12]:
import pandas as pd
from tqdm import tqdm

meta = pd.read_csv("/content/drive/MyDrive/copydays-ndid/NDID_6000_image_data_set/embeddings/clip_embeddings_meta.csv")
emb = np.load("/content/drive/MyDrive/copydays-ndid/NDID_6000_image_data_set/embeddings/clip_embeddings.npy").astype("float32")

index = faiss.read_index("ndid_faiss.index")

K = 10
correct = 0

for i in tqdm(range(len(emb))):
    q = emb[i].reshape(1, -1)
    D, I = index.search(q, K)

    q_group = meta.iloc[i]["group_id"]
    nn_groups = meta.iloc[I[0]]["group_id"].values

    if q_group in nn_groups[1:]:
        correct += 1

print("Recall@10:", correct / len(emb))


100%|██████████| 30000/30000 [03:55<00:00, 127.53it/s]

Recall@10: 0.9993333333333333
